In [11]:
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, KFold

import joblib

candidates = [
        (
            "baseline",
            DummyRegressor(strategy="median"),
            {},
        ),
        (
            "hist_gradient_boosting",
            HistGradientBoostingRegressor(
                random_state=42,
                early_stopping=True,
                max_iter=500,
                validation_fraction=0.15,
                scoring="loss",
                n_iter_no_change=10,
            ),
            {
                "learning_rate": [0.05, 0.1],
                "max_leaf_nodes": [15, 31],
                "min_samples_leaf": [5, 20],
                "l2_regularization": [0.0, 1.0],
            },
        ),
        (
            "random_forest",
            RandomForestRegressor(
                random_state=42,
                n_jobs=1,
            ),
            {
                "n_estimators": [100, 200],
                "max_depth": [None, 15],
                "min_samples_leaf": [1, 3],
            },
        ),
        (
            "extra_trees",
            ExtraTreesRegressor(
                random_state=42,
                n_jobs=1,
            ),
            {
                "n_estimators": [100, 200],
                "max_depth": [None, 15],
                "min_samples_leaf": [1, 3],
            },
        ),
    ]

def train_models():
    global candidates
    
    models = {}
    results = []
    
    cv = KFold(
        n_splits=3,
        shuffle=True,
        random_state=42,
    )
    
    X_train = pd.read_json("./train_data/X_train.json")
    X_test = pd.read_json("./train_data/X_test.json")
    y_train = pd.read_json("./train_data/y_train.json")
    y_test = pd.read_json("./train_data/y_test.json")
    
    models = {}
    results = []
    
    # predict a single column at a time across input features
    for target in y_train.columns:
        available = y_train[target].notna()
        target_X = X_train.loc[available]
        target_y = y_train.loc[available, target]
        
        if target_y.empty:
            print(f"{target}: skipped — no observed training values")
            continue
        
        best_model = None
        best_name = None
        best_params = {}
        best_cv_score = None
        
        for name, estimator, parameters in candidates:
            search = GridSearchCV(
                estimator=estimator, 
                param_grid=parameters, 
                scoring="neg_mean_absolute_error",
                cv=cv, 
                n_jobs=-1, 
                error_score="raise",
                verbose=2,
            )
            
            search.fit(target_X, target_y)

            print(
                f"{target} | {name} | "
                f"CV MAE: {-search.best_score_:.4f}",
                flush=True,
            )
            
            if best_cv_score is None or search.best_score_ > best_cv_score:
                best_model = search.best_estimator_
                best_name = name
                best_params = search.best_params_
                best_cv_score = search.best_score_
        models[target] = best_model
        
        available_test = y_test[target].notna()
        test_mae = None
        
        if available_test.any():
            test_predictions = best_model.predict(
                X_test.loc[available_test]
            )

            test_mae = mean_absolute_error(
                y_test.loc[available_test, target],
                test_predictions,
            )

        cv_mae = None
        if best_cv_score is not None:
            cv_mae = -best_cv_score

        results.append({
            "target": target,
            "model": best_name,
            "parameters": str(best_params),
            "cv_mae": cv_mae,
            "test_mae": test_mae,
        })

        print(
            f"Selected {best_name} for {target} | "
            f"Test MAE: {test_mae}",
            flush=True,
        )
    
    joblib.dump(models, "./model/models.joblib")
    
    results = pd.DataFrame(results)
    results.to_csv("./modelresults.csv", index=False)

In [ ]:
import time

def benchmark_models():
    models = joblib.load("./model/models.joblib")
    
    X_train = pd.read_json("./train_data/X_train.json")
    X_test = pd.read_json("./train_data/X_test.json")
    
    X = pd.concat([X_train, X_test])


    start = time.perf_counter()
    predicted_changes = pd.DataFrame(index=X.index)
    for target, model in models.items():
        predicted_changes[target] = model.predict(X)

    elapsed = time.perf_counter() - start

    print(f"Units: {len(X)}")
    print(f"Models: {len(models)}")
    print(f"Total prediction time: {elapsed:.4f} seconds")

    if len(X) > 0:
        print(f"Time per unit: {elapsed / len(X) * 1000:.4f} ms")
benchmark_models()

Units: 2802
Models: 37
Total prediction time: 1.5205 seconds
Time per unit: 0.5426 ms


In [ ]:
models = joblib.load("./model/models.joblib")
vanilla = pd.read_json("./cleaned_data/vanilla.json")

def predict_unit(unit_key):
    matches = vanilla["unit"].eq(unit_key)

    if not matches.any():
        raise ValueError(f"Unit not found: {unit_key}")

    unit_index = vanilla.index[matches][0]
    unit_input = X_processed.loc[[unit_index]]

    changes = {}

    for target, model in models.items():
        stat = target.removeprefix("delta_")
        changes[stat] = float(model.predict(unit_input)[0])

    return changes
predict_unit("wh2_dlc09_skv_cha_tretch_craventail")

In [12]:
if __name__ == "__main__":
    train_models()

Fitting 3 folds for each of 1 candidates, totalling 3 fits
delta_main_units__tier | baseline | CV MAE: 0.3396
Fitting 3 folds for each of 16 candidates, totalling 48 fits
delta_main_units__tier | hist_gradient_boosting | CV MAE: 0.2193
Fitting 3 folds for each of 8 candidates, totalling 24 fits
delta_main_units__tier | random_forest | CV MAE: 0.2163
Fitting 3 folds for each of 8 candidates, totalling 24 fits
delta_main_units__tier | extra_trees | CV MAE: 0.2086
Selected extra_trees for delta_main_units__tier | Test MAE: 0.19708110516934046
Fitting 3 folds for each of 1 candidates, totalling 3 fits
delta_main_units__num_men | baseline | CV MAE: 0.7407
Fitting 3 folds for each of 16 candidates, totalling 48 fits
delta_main_units__num_men | hist_gradient_boosting | CV MAE: 0.9492
Fitting 3 folds for each of 8 candidates, totalling 24 fits
delta_main_units__num_men | random_forest | CV MAE: 0.8384
Fitting 3 folds for each of 8 candidates, totalling 24 fits
delta_main_units__num_men | extra